In [1]:
# 提取案例库数据的标签
import pandas as pd
import json
import random
import re
import os

def read_json_files(folder_path):
    articles,charges,keys,facts,paths,node_ids = [],[],[],[],[],[]
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".json"):
                file_path = os.path.join(root, file)
                with open(file_path, 'r', encoding='utf-8') as f:
                    lines = f.readlines()
                    for line in lines:
                        line = json.loads(line)
                        article = line["art"]
                        charge = line["char"]
                        articles.append(article)
                        charges.append(charge)
                        keys.append(line["key2"])
                        facts.append(line["key"])
                        paths.append(line["path_list"])
                        node_ids.append(line["node_id"])
    return facts,keys,charges,articles,paths,node_ids
facts,keys,charges,articles,paths,node_ids = read_json_files("/root/data1/liang/self-correct-retriever/data/hera_knowbase3/法院观点")
know_df = pd.DataFrame()
know_df["fact"],know_df["key"],know_df["charge"],know_df["article"],know_df["path"],know_df["node_id"]=facts,keys,charges,articles,paths,node_ids
print(len(know_df))
know_df.head(2)


6124


,fact,key,charge,article,path,node_id
0,：2020年8月份开始，钟某某（另案处理）在惠州大亚湾**村组织多名卖淫女实施组织卖淫活动，...,"[组织卖淫活动, 介绍嫖客, 谋取非法利益, 介绍卖淫罪]",介绍卖淫罪,三百五十九,"[法院观点, 妨害社会管理秩序罪, 介绍卖淫罪]","[0, 6124, 12248]"
1,：\n\n2016年7月中旬，汤某某、诸某某伙同沈某某、张某某（均在逃）四人在西双版纳与缅甸...,"[违法事实, 未经出入境边防许可，利用乘坐摩托车的方式偷越国境, 自愿认罪认罚情节, 依据中...",偷越国(边)境罪,三百二十一,"[法院观点, 妨害社会管理秩序罪, 偷越国(边)境罪]","[1, 6125, 12249]"


In [2]:
know_df["fact"][0]

'：2020年8月份开始，钟某某（另案处理）在惠州大亚湾**村组织多名卖淫女实施组织卖淫活动，并在大亚湾**村**号（**店对面）租了两间房专门用于卖淫。2020年10月9日开始，李某某与钟某某约定好介绍嫖客的利润分成后就利用微信招嫖，拉到嫖客后就将嫖客介绍给钟某某，钟某某将卖淫女的房号告诉李某某后，李某某就告诉嫖客，由嫖客自行前往与卖淫女完成性交易。李某某为提高介绍卖淫的业绩，从10月22日起，其将添加过其微信的所有嫖客（有21名男嫖客），组建了一个微信群，并在群里发送新到的卖淫女照片，如果有需要嫖娼的，就可以主动联系其。经统计，李某某共介绍了五次嫖客给钟某某，其中有三人与卖淫女完成了性交易。\n\n认定上述。'

In [3]:
# 构造三元组，一个标签相同，一个不同，随机选
import pandas as pd
import json
import random
from tqdm import tqdm
with open("/root/data1/liang/self-correct-retriever/data/train_data/cvg_split/train.json","r") as f:
    data_all = []
    lines = f.readlines()
    random.shuffle(lines)
    for line in tqdm(lines):
        line = json.loads(line)
        charge,article = line["char"],line["art"]
        same_label_data = know_df[(know_df['charge'] == charge) & (know_df['article'] == article)]
        try:
            random_same_label_data = same_label_data.sample(n=1)
        except:
            continue
        different_label_data = know_df[(know_df['charge'] != charge) | (know_df['article'] != article)]
        try:
            random_different_label_data = different_label_data.sample(n=1)
        except:
            continue
        data_all.append({'origin': line["fact"],
                         'entailment': random_same_label_data.iloc[0]["fact"], 
                         'entailment_key':random_same_label_data.iloc[0]["key"], 
                         'entailment_path':random_same_label_data.iloc[0]["path"],
                         'entailment_node_id':random_same_label_data.iloc[0]["node_id"],
                         'contradiction': random_different_label_data.iloc[0]["fact"],
                         'contradiction_key':random_different_label_data.iloc[0]["key"],
                         'contradiction_path':random_different_label_data.iloc[0]["path"],
                         'contradiction_node_id':random_different_label_data.iloc[0]["node_id"]})
        
write_path = "/root/data1/liang/self-correct-retriever/data/knowledge_base/simcse_test_cvg.json"
with open(write_path, "w") as f:
    random.shuffle(data_all)
    test_ratio = int(len(data_all)*0.1)
    for dic in data_all[:test_ratio]:
        json.dump(dic,f,ensure_ascii=False)
        f.write("\n")
write_path = "/root/data1/liang/self-correct-retriever/data/knowledge_base/simcse_train_cvg.json"
with open(write_path, "w") as f:
    for dic in data_all[test_ratio:]:
        json.dump(dic,f,ensure_ascii=False)
        f.write("\n")
print(len(data_all))   
        

100%|██████████| 72308/72308 [04:11<00:00, 286.99it/s]


71955
